# Debug Data Quality

Investigate timestamp issues, wind error consistency, sentinel values, and outliers in `data/processed/all_data.parquet`.

In [ ]:
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_PATH = Path('data/processed/all_data.parquet')

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

df = pl.read_parquet(DATA_PATH)
df.shape

## Investigate Timestamps (Critical)

In [ ]:
# Null timestamps
if 'timestamp_utc' not in df.columns:
    raise KeyError('timestamp_utc not found in dataframe')

null_ts = df.filter(pl.col('timestamp_utc').is_null())
null_ts.height

In [ ]:
# Display rows with null timestamps
null_ts

In [ ]:
# Duplicate timestamps
dupe_keys = (
    df.filter(pl.col('timestamp_utc').is_not_null())
      .with_columns(pl.col('timestamp_utc').is_duplicated().alias('is_dupe'))
      .filter(pl.col('is_dupe'))
      .select('timestamp_utc')
      .unique()
)
dupe_keys.height

In [ ]:
# Show full rows for duplicate timestamps
dupe_rows = df.join(dupe_keys, on='timestamp_utc', how='inner').sort('timestamp_utc')
dupe_rows

## Investigate Wind Consistency

In [ ]:
cols_needed = {'wind_onshore_forecast', 'wind_onshore_actual', 'wind_onshore_error'}
missing = cols_needed - set(df.columns)
if missing:
    raise KeyError(f'Missing columns: {sorted(missing)}')

wind = df.select([
    'timestamp_utc',
    'wind_onshore_forecast',
    'wind_onshore_actual',
    'wind_onshore_error'
]).with_columns(
    (pl.col('wind_onshore_forecast') - pl.col('wind_onshore_actual')).alias('calc_error'),
).with_columns(
    (pl.col('calc_error') - pl.col('wind_onshore_error')).alias('error_diff')
)
wind.head()

In [ ]:
# Plot error difference distribution
plt.figure(figsize=(10, 4))
sns.histplot(wind['error_diff'].drop_nulls().to_numpy(), bins=80)
plt.title('Wind onshore error diff: calc_error - wind_onshore_error')
plt.xlabel('MW')
plt.ylabel('Count')
plt.tight_layout()

In [ ]:
# Rows where difference > 100 MW
wind_bad = wind.filter(pl.col('error_diff').abs() > 100)
wind_bad.height

In [ ]:
wind_bad.sort('timestamp_utc').head(50)

## Investigate Sentinel Values (99999)

In [ ]:
cols_prices = [c for c in ['afrr_activation_price_pos', 'afrr_activation_price_neg'] if c in df.columns]
if not cols_prices:
    raise KeyError('afrr_activation_price_pos/neg not found')

sentinel_filter = None
for c in cols_prices:
    expr = (pl.col(c) > 10000) | (pl.col(c) < -10000)
    sentinel_filter = expr if sentinel_filter is None else (sentinel_filter | expr)

sentinel_rows = df.filter(sentinel_filter)
sentinel_rows.height

In [ ]:
# Show representative sentinel rows
sentinel_rows.select(['timestamp_utc'] + cols_prices).sort('timestamp_utc').head(50)

## Investigate Outliers

In [ ]:
if 'total_wind_intraday_error' not in df.columns:
    raise KeyError('total_wind_intraday_error not found')

max_row = df.sort('total_wind_intraday_error', descending=True).head(1)
max_row

In [ ]:
# Check related wind values (if present)
wind_cols = [c for c in [
    'wind_onshore_actual', 'wind_onshore_forecast',
    'wind_offshore_actual', 'wind_offshore_forecast',
    'total_wind_actual', 'total_wind_forecast'
] if c in df.columns]
max_row.select(['timestamp_utc', 'total_wind_intraday_error'] + wind_cols)